# Tutotial RAG with DSPY

In [9]:
from typing import List

In [19]:
matrix = [[1, 3]]
target = 3

def searchMatrix(matrix: List[List[int]], target: int) -> bool:
        for i, r in enumerate(matrix):
            if r[0] > target:
                    for e in matrix[i-1]:
                        if e == target:
                            return True
                    return False
            elif r[0] == target:
                return True
        
        return False
searchMatrix(matrix, target)

False

## Configure env

In [5]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5555")
mlflow.set_experiment("DSPy")

2025/05/16 17:06:55 INFO mlflow.tracking.fluent: Experiment with name 'DSPy' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/977266803468962292', creation_time=1747408014999, experiment_id='977266803468962292', last_update_time=1747408014999, lifecycle_stage='active', name='DSPy', tags={}>

In [6]:
mlflow.dspy.autolog()

In [7]:
import dspy
import os
from dotenv import load_dotenv
load_dotenv()

# get env vars 
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")
api_key = os.getenv("AZURE_OPENAI_KEY")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")

lm = dspy.LM(f"azure/{deployment}", api_key=api_key, api_base=endpoint)
dspy.configure(lm=lm)

## Basics

Modules: they take a signature (structured input/output schema) and givexs you back a func for the behavior specified. 

In [8]:
qa = dspy.Predict('question: str -> response: str')
response = qa(question="what are high memory and low memory on Linux?")
print(response.response)

17:15:32 - LiteLLM:INFO: utils.py:2900 - 
LiteLLM completion() model= gpt35-sr-agent; provider = azure
INFO:LiteLLM:
LiteLLM completion() model= gpt35-sr-agent; provider = azure
INFO:httpx:HTTP Request: POST https://oai-h-ai4sr486387875618.openai.azure.com/openai/deployments/gpt35-sr-agent/chat/completions?api-version=2025-02-01-preview "HTTP/1.1 200 OK"
17:15:33 - LiteLLM:INFO: utils.py:1209 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
17:15:33 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-3.5-turbo-0125
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-3.5-turbo-0125
17:15:33 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-3.5-turbo-0125
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-3.5-turbo-0125
17:15:33 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure

In Linux, "high memory" refers to the portion of memory above the 4GB limit in a 32-bit system, while "low memory" refers to the portion of memory below the 4GB limit. High memory is typically used for kernel code and data structures, while low memory is used for user-space processes.


Trace(request_id=54dfcd7abad4420397c8b175b7e468c0)

17:22:23 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-3.5-turbo-0125
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-3.5-turbo-0125
17:22:23 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-3.5-turbo-0125
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-3.5-turbo-0125


We have different modules:

- .ChainOfThought
- .PrograOfThough
- .ReAct


In [10]:
# example of chain of thought
cot = dspy.ChainOfThought('question -> response')
cot(question='Are low-level coding languages always better than high-level ones?')

17:22:21 - LiteLLM:INFO: utils.py:2900 - 
LiteLLM completion() model= gpt35-sr-agent; provider = azure
INFO:LiteLLM:
LiteLLM completion() model= gpt35-sr-agent; provider = azure
INFO:httpx:HTTP Request: POST https://oai-h-ai4sr486387875618.openai.azure.com/openai/deployments/gpt35-sr-agent/chat/completions?api-version=2025-02-01-preview "HTTP/1.1 200 OK"
17:22:23 - LiteLLM:INFO: utils.py:1209 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
17:22:23 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-3.5-turbo-0125
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-3.5-turbo-0125


Prediction(
    reasoning='Low-level coding languages and high-level coding languages each have their own advantages and disadvantages. Low-level languages provide more direct control over hardware and memory, making them more efficient in terms of speed and memory usage. On the other hand, high-level languages are more user-friendly, easier to learn, and generally more productive for software development.',
    response='Low-level coding languages are not always better than high-level ones. The choice between low-level and high-level languages depends on the specific requirements of the project, the level of control needed, and the trade-offs between efficiency and productivity.'
)

Trace(request_id=4e6413f25189475fa6162fe02916af37)

## Optimization DSPy